# Traditional ML Synthetic Options Backtest
This notebook trains one Random Forest classifier for each technical family, one classifier for each selected FMP feature family (including separate key metrics, ratios, income statement, cash flow, and balance sheet TTM models), separate historical sector-performance, industry-performance, sector-P/E, and industry-P/E classifiers, and one macro classifier over economic indicators plus treasury rates. It then combines those classifiers as a feature-family mixture of experts (MoE), scores the configured stock universe, and compares the resulting classifier-only trading signal across equity and synthetic constant-maturity option proxies.

Instrument variants:
- equity
- ATM call/put proxy
- OTM call/put proxy
- DITM call/put proxy

Strategy variants:
- `top_k = 5`
- `top_k = 10`
- `top_k = 20`
- `top_k = 40`

Synthetic approximation assumptions:
- constant-maturity option index, rolled continuously each day
- Black-Scholes pricing with realized-volatility proxy
- ATM = strike multiplier `1.00`
- OTM = call strike multiplier `1.05`, put strike multiplier `0.95`
- DITM = call strike multiplier `0.90`, put strike multiplier `1.10`
- tenor fixed at `60` trading days

This is a research approximation, not a broker-realistic options backtest.


In [1]:
import gc
import importlib.util
import json
import math
import os
import pickle
import time
from pathlib import Path
from contextlib import contextmanager
from concurrent.futures import ThreadPoolExecutor, as_completed
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from scipy.special import ndtr as _scipy_ndtr
except Exception:
    _scipy_ndtr = None
try:
    import cupy as cp
except Exception:
    cp = None
from IPython.display import display

import django
from django.apps import apps
from django.db import close_old_connections

os.environ.setdefault("DJANGO_ALLOW_ASYNC_UNSAFE", "true")
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "settings")
if not apps.ready:
    django.setup()

from fmp.workflows import run_scoring_data_refresh_from_fmp
from fmp.endpoints.helpers import HISTORICAL_TARGET_MIN_DATE
from fmp.classification_performance import refresh_classification_performance
from fmp.classification_pe import refresh_classification_pe
from fmp.models import Symbol
from features.feature_builders import (
    build_industry_performance_features,
    build_industry_pe_features,
    build_price_technical_features,
    build_sector_performance_features,
    build_sector_pe_features,
    build_ta_classic_technical_features,
)
from features.balance_sheet_features import build_balance_sheet_features
from features.balance_sheet_growth_features import build_balance_sheet_growth_features
from features.cash_flow_features import build_cash_flow_features
from features.cash_flow_growth_features import build_cash_flow_growth_features
from features.earnings_features import build_earnings_features
from features.financial_growth_features import build_financial_growth_features
from features.income_statement_features import build_income_statement_features
from features.income_statement_growth_features import build_income_statement_growth_features
from features.key_metrics_features import build_key_metrics_features
from features.ratios_features import build_ratios_features
from features.section_utils import clear_section_record_cache, prime_section_record_cache
from features.views import _load_adjusted_prices
from pipeline.api import build_macro_dataframe, build_label_dataframe
from data.preparation import MLDatasetConfig, prepare_ml_dataset
from ml.base import FitSpec
from ml.frameworks.sklearn import CumlRFClassifier, SklearnRFClassifier
from backtest.raw_stack import ProbabilityColumnConfig, enrich_scored_panel, make_backtest_panel
from workflows.fmp_feature_families import build_fmp_endpoint_feature_families as shared_build_fmp_endpoint_feature_families
from workflows.strategy import backtest_positions_with_directional_asset_returns as shared_backtest_positions_with_directional_asset_returns
from workflows.strategy import prepare_backtest_position_state as shared_prepare_backtest_position_state
from workflows.options_pricing import build_constant_maturity_call_price_panel as shared_build_constant_maturity_call_price_panel
from workflows.options_pricing import build_constant_maturity_put_price_panel as shared_build_constant_maturity_put_price_panel
from workflows.options_pricing import build_realized_vol_panel as shared_build_realized_vol_panel
from pipeline.universe_selection import DEFAULT_US_EXCHANGES, resolve_symbol_universe
from data.universe_fmp import resolve_symbol_universe_from_screener
from fmp.refresh import resolve_fmp_api_key

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)


def cuda_available():
    if cp is None:
        return False
    try:
        return int(cp.cuda.runtime.getDeviceCount()) > 0
    except Exception:
        return False


def cuml_available():
    if not cuda_available():
        return False
    try:
        return importlib.util.find_spec("cuml") is not None
    except Exception:
        return False


CUDA_AVAILABLE = cuda_available()
CUML_AVAILABLE = cuml_available()
print(f"CUDA available: {CUDA_AVAILABLE} | cuML available: {CUML_AVAILABLE}")


def current_memory_mb():
    try:
        import psutil
        return float(psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2))
    except Exception:
        try:
            import resource
            usage = float(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
            # macOS reports ru_maxrss in bytes; Linux usually reports KB.
            return usage / (1024 ** 2) if usage > 10_000_000 else usage / 1024.0
        except Exception:
            return float("nan")


def log_memory(label, *, rows=None, cols=None, collect=False):
    if collect:
        gc.collect()
    rss_mb = current_memory_mb()
    details = []
    if rows is not None:
        details.append(f"rows={int(rows):,}")
    if cols is not None:
        details.append(f"cols={int(cols):,}")
    detail_text = " | " + " | ".join(details) if details else ""
    print(f"[memory] {label}: rss={rss_mb:,.1f} MB{detail_text}")
    return rss_mb


def log_refresh(message):
    ts = pd.Timestamp.now(tz="America/Los_Angeles").strftime("%H:%M:%S")
    print(f"[{ts}] {message}", flush=True)


PHASE_TIMINGS = []


@contextmanager
def timed_phase(label, *, rows=None, cols=None, collect_before=False, collect_after=True):
    if collect_before:
        gc.collect()
    start_time = time.perf_counter()
    start_mem = current_memory_mb()
    print(f"[timer] start {label} | rss={start_mem:,.1f} MB", flush=True)
    try:
        yield
    finally:
        if collect_after:
            gc.collect()
        elapsed = time.perf_counter() - start_time
        end_mem = current_memory_mb()
        details = []
        if rows is not None:
            details.append(f"rows={int(rows):,}")
        if cols is not None:
            details.append(f"cols={int(cols):,}")
        detail_text = " | " + " | ".join(details) if details else ""
        print(
            f"[timer] done  {label} | elapsed={elapsed:,.2f}s"
            f" | rss={end_mem:,.1f} MB | delta={end_mem - start_mem:+,.1f} MB{detail_text}",
            flush=True,
        )
        PHASE_TIMINGS.append({
            "phase": str(label),
            "elapsed_sec": float(elapsed),
            "rss_start_mb": float(start_mem),
            "rss_end_mb": float(end_mem),
            "rss_delta_mb": float(end_mem - start_mem),
            "rows": int(rows) if rows is not None else None,
            "cols": int(cols) if cols is not None else None,
        })



def _build_technical_dataframe_from_django(*, symbols, start_date=None, end_date=None):
    start_ts = pd.Timestamp(start_date) if start_date is not None else None
    end_ts = pd.Timestamp(end_date) if end_date is not None else None
    frames = []
    feature_cols = []
    technical_family_frames = {}
    technical_family_cols = {}
    for sym in symbols:
        code = str(sym).strip().upper()
        if not code:
            continue

        symbol_obj = Symbol.objects.filter(symbol__iexact=code).only("id", "symbol").first()
        if symbol_obj is None:
            continue

        df_prices = _load_adjusted_prices(
            symbol_obj,
            start_ts.date() if start_ts is not None else None,
            end_ts.date() if end_ts is not None else None,
        )
        if df_prices.empty:
            continue

        built = build_price_technical_features(code, df_prices)
        if built.df.empty:
            continue

        px = df_prices[["open", "high", "low", "close", "volume"]].copy()
        px["symbol"] = code
        px = px.reset_index().set_index(["date", "symbol"]).sort_index()

        panel = px.join(built.df[built.feature_cols], how="left")
        frames.append(panel)

        for family_name, family_built in build_ta_classic_technical_features(code, df_prices).items():
            active_cols = [
                col
                for col in family_built.feature_cols
                if col in family_built.df.columns and pd.api.types.is_numeric_dtype(family_built.df[col])
            ]
            if not active_cols:
                continue
            technical_family_frames.setdefault(family_name, []).append(family_built.df.loc[:, active_cols])
            family_cols = technical_family_cols.setdefault(family_name, [])
            for col in active_cols:
                if col not in family_cols:
                    family_cols.append(col)
        for col in built.feature_cols:
            if col not in feature_cols:
                feature_cols.append(col)

    if not frames:
        empty_index = pd.MultiIndex(levels=[[], []], codes=[[], []], names=["date", "symbol"])
        return pd.DataFrame(index=empty_index), feature_cols, {}, {}

    technical_df = pd.concat(frames, axis=0).sort_index()
    if technical_df.index.has_duplicates:
        technical_df = technical_df[~technical_df.index.duplicated(keep="last")]
    split_family_frames = {}
    split_family_cols = {}
    for family_name, family_frames in technical_family_frames.items():
        family_frame = pd.concat(family_frames, axis=0).sort_index()
        if family_frame.index.has_duplicates:
            family_frame = family_frame[~family_frame.index.duplicated(keep="last")]
        cols = [c for c in technical_family_cols.get(family_name, []) if c in family_frame.columns]
        cols = [c for c in cols if pd.api.types.is_numeric_dtype(family_frame[c]) and family_frame[c].notna().any()]
        if cols:
            split_family_frames[family_name] = family_frame.loc[:, cols].astype(np.float32, copy=False)
            split_family_cols[family_name] = cols
    return technical_df, feature_cols, split_family_frames, split_family_cols


def _target_index_for_symbol(target_index, symbol):
    code = str(symbol).strip().upper()
    mask = target_index.get_level_values("symbol").astype(str).str.upper() == code
    dates = pd.DatetimeIndex(pd.to_datetime(target_index.get_level_values("date")[mask])).normalize()
    return pd.MultiIndex.from_arrays([dates, [code] * len(dates)], names=["date", "symbol"])


def _price_frame_for_symbol(price_panel, symbol):
    code = str(symbol).strip().upper()
    try:
        return price_panel.xs(code, level="symbol")
    except Exception:
        return pd.DataFrame()


def _build_classification_performance_feature_families(*, symbols, target_index, price_panel, progress_logger=None):
    builders = {
        "sector_performance": lambda obj, idx, px: build_sector_performance_features(obj, idx, df_prices=px),
        "industry_performance": lambda obj, idx, px: build_industry_performance_features(obj, idx, df_prices=px),
        "sector_pe": lambda obj, idx, px: build_sector_pe_features(obj, idx),
        "industry_pe": lambda obj, idx, px: build_industry_pe_features(obj, idx),
    }
    normalized_symbols = [str(symbol).strip().upper() for symbol in symbols if str(symbol).strip()]
    symbol_objs = {
        str(obj.symbol).strip().upper(): obj
        for obj in Symbol.objects.filter(symbol__in=normalized_symbols).only(
            "id", "symbol", "exchange", "sector", "industry"
        )
    }
    family_parts = {name: [] for name in builders}
    family_cols = {name: [] for name in builders}
    total = len(normalized_symbols)
    for position, code in enumerate(normalized_symbols, start=1):
        symbol_obj = symbol_objs.get(code)
        if symbol_obj is None:
            continue
        symbol_index = _target_index_for_symbol(target_index, code)
        if len(symbol_index) == 0:
            continue
        symbol_prices = _price_frame_for_symbol(price_panel, code)
        for family_name, builder in builders.items():
            built = builder(symbol_obj, symbol_index, symbol_prices)
            active_cols = [
                col for col in built.feature_cols
                if col in built.df.columns and pd.api.types.is_numeric_dtype(built.df[col])
            ]
            if not active_cols:
                continue
            family_parts[family_name].append(built.df.loc[:, active_cols])
            for col in active_cols:
                if col not in family_cols[family_name]:
                    family_cols[family_name].append(col)
        if callable(progress_logger) and (position == 1 or position % 100 == 0 or position == total):
            progress_logger(
                f"Classification performance feature build: {position:,}/{total:,} symbols processed"
            )
    family_frames = {}
    active_family_cols = {}
    for family_name, parts in family_parts.items():
        if not parts:
            continue
        frame = pd.concat(parts, axis=0).sort_index()
        if frame.index.has_duplicates:
            frame = frame.loc[~frame.index.duplicated(keep="last")]
        cols = [col for col in family_cols[family_name] if col in frame.columns and frame[col].notna().any()]
        if not cols:
            continue
        family_frames[family_name] = frame.loc[:, cols].astype(np.float32, copy=False)
        active_family_cols[family_name] = cols
    return family_frames, active_family_cols


def _fmp_endpoint_builders(filing_lag_days):
    return {
        "key_metrics": lambda symbol_obj, idx, px, market_cap, valuation_context: build_key_metrics_features(symbol_obj, idx, df_prices=px, filing_lag_days=filing_lag_days),
        "ratios": lambda symbol_obj, idx, px, market_cap, valuation_context: build_ratios_features(symbol_obj, idx, df_prices=px, filing_lag_days=filing_lag_days),
        "income_statement": lambda symbol_obj, idx, px, market_cap, valuation_context: build_income_statement_features(symbol_obj, idx, df_prices=px, filing_lag_days=filing_lag_days),
        "income_statement_growth": lambda symbol_obj, idx, px, market_cap, valuation_context: build_income_statement_growth_features(symbol_obj, idx, valuation_frame=valuation_context, filing_lag_days=filing_lag_days),
        "balance_sheet": lambda symbol_obj, idx, px, market_cap, valuation_context: build_balance_sheet_features(symbol_obj, idx, df_prices=px, market_cap=market_cap, filing_lag_days=filing_lag_days),
        "balance_sheet_growth": lambda symbol_obj, idx, px, market_cap, valuation_context: build_balance_sheet_growth_features(symbol_obj, idx, valuation_frame=valuation_context, filing_lag_days=filing_lag_days),
        "cash_flow": lambda symbol_obj, idx, px, market_cap, valuation_context: build_cash_flow_features(symbol_obj, idx, df_prices=px, market_cap=market_cap, filing_lag_days=filing_lag_days),
        "cash_flow_growth": lambda symbol_obj, idx, px, market_cap, valuation_context: build_cash_flow_growth_features(symbol_obj, idx, valuation_frame=valuation_context, filing_lag_days=filing_lag_days),
        "financial_growth": lambda symbol_obj, idx, px, market_cap, valuation_context: build_financial_growth_features(symbol_obj, idx, valuation_frame=valuation_context, filing_lag_days=filing_lag_days),
        "earnings": lambda symbol_obj, idx, px, market_cap, valuation_context: build_earnings_features(symbol_obj, idx),
    }


def _build_fmp_endpoint_features_for_symbol(code, symbol_obj, *, target_index, price_panel, endpoint_builders):
    close_old_connections()
    try:
        symbol_index = _target_index_for_symbol(target_index, code)
        if len(symbol_index) == 0:
            return code, {}, {}, "empty_index", 0.0
        symbol_prices = _price_frame_for_symbol(price_panel, code)
        symbol_market_cap = None
        valuation_context = pd.DataFrame(index=symbol_index)
        frames = {}
        cols_by_endpoint = {}
        start_time = time.perf_counter()
        for endpoint_name, builder in endpoint_builders.items():
            built = builder(symbol_obj, symbol_index, symbol_prices, symbol_market_cap, valuation_context)
            active_cols = [c for c in built.feature_cols if c in built.df.columns and pd.api.types.is_numeric_dtype(built.df[c])]
            if endpoint_name == "key_metrics" and "km__marketcap" in built.df.columns:
                symbol_market_cap = pd.to_numeric(built.df["km__marketcap"], errors="coerce")
            if endpoint_name in {"key_metrics", "ratios", "income_statement", "balance_sheet", "cash_flow"} and not built.df.empty:
                valuation_context = pd.concat([valuation_context, built.df], axis=1)
                if valuation_context.columns.has_duplicates:
                    valuation_context = valuation_context.loc[:, ~valuation_context.columns.duplicated(keep="last")]
            if not active_cols:
                continue
            frames[endpoint_name] = built.df[active_cols]
            cols_by_endpoint[endpoint_name] = list(active_cols)
        return code, frames, cols_by_endpoint, "ok", time.perf_counter() - start_time
    except Exception as exc:
        return code, {}, {}, f"error:{type(exc).__name__}: {exc}", 0.0
    finally:
        close_old_connections()


def _build_fmp_endpoint_feature_families(*, symbols, target_index, price_panel, filing_lag_days=45, max_workers=1, progress_logger=None):
    endpoint_builders = _fmp_endpoint_builders(filing_lag_days)
    normalized_symbols = [str(s).strip().upper() for s in symbols if str(s).strip()]
    symbol_objs = {
        str(obj.symbol).strip().upper(): obj
        for obj in Symbol.objects.filter(symbol__in=normalized_symbols)
    }
    endpoint_frames = {name: [] for name in endpoint_builders}
    endpoint_cols = {name: [] for name in endpoint_builders}
    total = len(normalized_symbols)
    workers = max(1, int(max_workers or 1))
    errors = []

    def _record_result(result, idx):
        code, frames, cols_by_endpoint, status, elapsed = result
        if str(status).startswith("error:"):
            errors.append((code, status))
        for endpoint_name, frame in frames.items():
            endpoint_frames[endpoint_name].append(frame)
            for col in cols_by_endpoint.get(endpoint_name, []):
                if col not in endpoint_cols[endpoint_name]:
                    endpoint_cols[endpoint_name].append(col)
        if callable(progress_logger) and (idx == 1 or idx % 25 == 0 or idx == total or str(status).startswith("error:")):
            progress_logger(
                f"FMP feature build progress: {idx:,}/{total:,} symbols processed"
                f" | latest={code} | status={status} | elapsed={elapsed:.2f}s"
            )

    try:
        prime_section_record_cache(list(symbol_objs.values()), list(endpoint_builders.keys()))
        if callable(progress_logger):
            progress_logger(f"FMP feature build start | symbols={total:,} | workers={workers:,} | families={len(endpoint_builders):,}")
        if workers == 1:
            for idx, code in enumerate(normalized_symbols, start=1):
                symbol_obj = symbol_objs.get(code) or Symbol.objects.filter(symbol__iexact=code).first()
                if symbol_obj is None:
                    _record_result((code, {}, {}, "missing_symbol", 0.0), idx)
                    continue
                result = _build_fmp_endpoint_features_for_symbol(
                    code,
                    symbol_obj,
                    target_index=target_index,
                    price_panel=price_panel,
                    endpoint_builders=endpoint_builders,
                )
                _record_result(result, idx)
        else:
            futures = []
            with ThreadPoolExecutor(max_workers=workers) as executor:
                for code in normalized_symbols:
                    symbol_obj = symbol_objs.get(code)
                    if symbol_obj is None:
                        continue
                    futures.append(
                        executor.submit(
                            _build_fmp_endpoint_features_for_symbol,
                            code,
                            symbol_obj,
                            target_index=target_index,
                            price_panel=price_panel,
                            endpoint_builders=endpoint_builders,
                        )
                    )
                completed = 0
                for future in as_completed(futures):
                    completed += 1
                    _record_result(future.result(), completed)
    finally:
        clear_section_record_cache()

    family_frames = {}
    family_cols = {}
    for endpoint_name, frames in endpoint_frames.items():
        if not frames:
            continue
        with timed_phase(f"assemble FMP feature family {endpoint_name}", rows=sum(len(frame) for frame in frames), collect_after=True):
            frame = pd.concat(frames, axis=0).sort_index()
            if frame.index.has_duplicates:
                frame = frame[~frame.index.duplicated(keep="last")]
            cols = [c for c in endpoint_cols[endpoint_name] if c in frame.columns and pd.api.types.is_numeric_dtype(frame[c])]
            cols = [c for c in cols if frame[c].notna().any()]
            if cols:
                frame = frame.loc[:, cols].astype(np.float32, copy=False)
                family_frames[endpoint_name] = frame
                family_cols[endpoint_name] = cols
    if errors and callable(progress_logger):
        progress_logger(f"FMP feature build completed with {len(errors):,} symbol errors | first={errors[0]}")
    return family_frames, family_cols


def summarize_curve(returns, years, mode):
    returns = pd.Series(returns).fillna(0.0)
    equity = (1.0 + returns).cumprod()
    total_return_pct = float((equity.iloc[-1] - 1.0) * 100.0) if len(equity) else np.nan
    sharpe = float((returns.mean() / returns.std(ddof=0)) * np.sqrt(252.0)) if len(returns) and returns.std(ddof=0) > 1e-12 else np.nan
    max_drawdown_pct = float((((equity / equity.cummax()) - 1.0).min()) * 100.0) if len(equity) else np.nan
    yearly_rows = []
    for yr in years:
        yret = returns.loc[(returns.index >= pd.Timestamp(f"{yr}-01-01")) & (returns.index <= pd.Timestamp(f"{yr}-12-31"))]
        yeq = (1.0 + yret).cumprod()
        yearly_rows.append(
            {
                "mode": str(mode),
                "test_year": int(yr),
                "total_return_pct": float((yeq.iloc[-1] - 1.0) * 100.0) if len(yeq) else np.nan,
                "sharpe": float((yret.mean() / yret.std(ddof=0)) * np.sqrt(252.0)) if len(yret) and yret.std(ddof=0) > 1e-12 else np.nan,
                "max_drawdown_pct": float((((yeq / yeq.cummax()) - 1.0).min()) * 100.0) if len(yeq) else np.nan,
            }
        )
    return {
        "total_return_pct": total_return_pct,
        "sharpe": sharpe,
        "max_drawdown_pct": max_drawdown_pct,
        "equity_curve": equity,
        "yearly_df": pd.DataFrame(yearly_rows),
    }


def _pivot_rule_panel(panel, col, *, symbols=None):
    working_symbols = sorted(panel.index.get_level_values("symbol").unique()) if symbols is None else list(symbols)
    work = panel[[col]].reset_index()
    if work.duplicated(subset=["date", "symbol"]).any():
        work = (
            work.sort_values(["date", "symbol"])
            .groupby(["date", "symbol"], as_index=False, sort=False)
            .last()
        )
    return (
        work
        .pivot(index="date", columns="symbol", values=col)
        .reindex(columns=working_symbols)
        .sort_index()
    )


def _prepare_capacity_rule_inputs(panel, score_col, component_cols, price_col):
    if panel.index.has_duplicates:
        panel = panel[~panel.index.duplicated(keep="last")]
    symbols = sorted(panel.index.get_level_values("symbol").unique())
    score = _pivot_rule_panel(panel, score_col, symbols=symbols).shift(1)
    prob_buy = _pivot_rule_panel(panel, "prob_buy", symbols=symbols).shift(1)
    prob_short = _pivot_rule_panel(panel, "prob_short", symbols=symbols).shift(1)
    close = _pivot_rule_panel(panel, price_col, symbols=symbols)
    common_dates = score.index.intersection(prob_buy.index).intersection(prob_short.index).intersection(close.index)
    score = score.loc[common_dates].replace([np.inf, -np.inf], np.nan)
    prob_buy = prob_buy.loc[common_dates].replace([np.inf, -np.inf], np.nan)
    prob_short = prob_short.loc[common_dates].replace([np.inf, -np.inf], np.nan)
    close = close.loc[common_dates].replace([np.inf, -np.inf], np.nan).ffill().fillna(0.0)
    component_frames = {}
    for col in component_cols:
        component = _pivot_rule_panel(panel, col, symbols=symbols).shift(1).reindex(index=common_dates, columns=symbols)
        component_frames[str(col)] = component.replace([np.inf, -np.inf], np.nan)
    return {
        "symbols": symbols,
        "common_dates": common_dates,
        "score": score,
        "prob_buy": prob_buy,
        "prob_short": prob_short,
        "close": close,
        "component_cols": [str(col) for col in component_cols],
        "component_frames": component_frames,
    }


def _build_entry_ok_matrix(inputs, component_threshold):
    score = inputs["score"]
    close = inputs["close"]
    entry_ok = (score.notna() & np.isfinite(score) & close.gt(0.0)).fillna(False)
    for col in inputs["component_cols"]:
        component = inputs["component_frames"][col]
        component_valid = component.notna() & np.isfinite(component)
        entry_ok &= (component.gt(float(component_threshold)) & component_valid).fillna(False)
    return entry_ok


def _run_capacity_limited_long_only_rule(*, panel, score_col, component_cols, component_threshold, price_col, top_k=None):
    inputs = _prepare_capacity_rule_inputs(panel, score_col, component_cols, price_col)
    symbols = inputs["symbols"]
    common_dates = inputs["common_dates"]
    score = inputs["score"]
    prob_buy = inputs["prob_buy"]
    prob_short = inputs["prob_short"]
    close = inputs["close"]

    entry_ok = _build_entry_ok_matrix(inputs, component_threshold)

    held_idx = set()
    symbol_to_idx = {sym: idx for idx, sym in enumerate(symbols)}
    position_by_day = pd.DataFrame(0, index=common_dates, columns=symbols, dtype=int)

    for dt in common_dates:
        prob_buy_t = prob_buy.loc[dt]
        prob_short_t = prob_short.loc[dt]
        classifier_short = (prob_short_t > prob_buy_t).fillna(False)

        exit_idx = sorted(
            idx
            for idx in held_idx
            if bool(classifier_short.iloc[idx])
        )
        if exit_idx:
            held_idx -= set(exit_idx)

        slots_left = None if top_k is None else max(0, int(top_k) - len(held_idx))
        if slots_left != 0:
            price_ok_t = close.loc[dt].gt(0.0)
            candidate_mask = entry_ok.loc[dt] & price_ok_t & (~classifier_short)
            ranked = score.loc[dt][candidate_mask].sort_values(ascending=False, kind="stable")
            enter_idx = []
            for sym in ranked.index:
                idx = symbol_to_idx[str(sym)]
                if idx in held_idx:
                    continue
                enter_idx.append(idx)
                if slots_left is not None and len(enter_idx) >= slots_left:
                    break
            if enter_idx:
                held_idx |= set(enter_idx)

        if held_idx:
            position_by_day.loc[dt, [symbols[idx] for idx in sorted(held_idx)]] = 1

    return {
        "positions": position_by_day,
        "score": score,
        "close": close,
    }


def run_top_k_long_only_score_rule(*, panel, score_col, component_cols, component_threshold, price_col, top_k, rebalance_freq=None):
    _ = rebalance_freq
    return _run_capacity_limited_long_only_rule(
        panel=panel,
        score_col=score_col,
        component_cols=component_cols,
        component_threshold=component_threshold,
        price_col=price_col,
        top_k=int(top_k),
    )


def _run_capacity_limited_long_short_rule(
    *,
    panel,
    long_score_col,
    short_score_col,
    long_component_cols,
    short_component_cols,
    component_threshold,
    price_col,
    top_k=None,
):
    long_inputs = _prepare_capacity_rule_inputs(panel, long_score_col, long_component_cols, price_col)
    short_inputs = _prepare_capacity_rule_inputs(panel, short_score_col, short_component_cols, price_col)
    symbols = long_inputs["symbols"]
    common_dates = long_inputs["common_dates"]
    close = long_inputs["close"]
    long_score = long_inputs["score"]
    short_score = short_inputs["score"]
    prob_buy = long_inputs["prob_buy"]
    prob_short = long_inputs["prob_short"]

    long_entry_ok = _build_entry_ok_matrix(long_inputs, component_threshold)
    short_entry_ok = _build_entry_ok_matrix(short_inputs, component_threshold)

    held_side_by_idx = {}
    symbol_to_idx = {sym: idx for idx, sym in enumerate(symbols)}
    position_by_day = pd.DataFrame(0, index=common_dates, columns=symbols, dtype=int)

    for dt in common_dates:
        price_ok_t = close.loc[dt].gt(0.0)
        prob_buy_t = prob_buy.loc[dt]
        prob_short_t = prob_short.loc[dt]
        long_score_t = long_score.loc[dt]
        short_score_t = short_score.loc[dt]

        next_held = {}
        for idx, side in sorted(held_side_by_idx.items()):
            if side > 0:
                if bool(prob_short_t.iloc[idx] > prob_buy_t.iloc[idx]):
                    continue
            else:
                if bool(prob_buy_t.iloc[idx] > prob_short_t.iloc[idx]):
                    continue
            next_held[idx] = side
        held_side_by_idx = next_held

        capacity = None if top_k is None else max(0, int(top_k))
        slots_left = None if capacity is None else max(0, capacity - len(held_side_by_idx))
        if slots_left != 0:
            candidates = []
            for sym in symbols:
                idx = symbol_to_idx[str(sym)]
                if idx in held_side_by_idx or (not bool(price_ok_t.iloc[idx])):
                    continue
                long_ok = bool(long_entry_ok.loc[dt].iloc[idx]) and np.isfinite(long_score_t.iloc[idx])
                short_ok = bool(short_entry_ok.loc[dt].iloc[idx]) and np.isfinite(short_score_t.iloc[idx])
                if not long_ok and not short_ok:
                    continue
                if long_ok and short_ok:
                    long_value = float(long_score_t.iloc[idx])
                    short_value = float(short_score_t.iloc[idx])
                    if long_value >= short_value:
                        best_side, best_score = 1, long_value
                    else:
                        best_side, best_score = -1, short_value
                elif long_ok:
                    best_side, best_score = 1, float(long_score_t.iloc[idx])
                else:
                    best_side, best_score = -1, float(short_score_t.iloc[idx])
                candidates.append((best_score, str(sym), idx, best_side))

            for _score_value, _sym, idx, side in sorted(candidates, key=lambda row: (row[0], row[1]), reverse=True):
                held_side_by_idx[idx] = int(side)
                if slots_left is not None and len(held_side_by_idx) >= int(capacity):
                    break

        for idx, side in sorted(held_side_by_idx.items()):
            position_by_day.loc[dt, symbols[idx]] = int(side)

    return {
        "positions": position_by_day,
        "long_score": long_score,
        "short_score": short_score,
        "close": close,
    }


def run_top_k_long_short_score_rule(
    *,
    panel,
    long_score_col,
    short_score_col,
    long_component_cols,
    short_component_cols,
    component_threshold,
    price_col,
    top_k,
    rebalance_freq=None,
):
    _ = rebalance_freq
    return _run_capacity_limited_long_short_rule(
        panel=panel,
        long_score_col=long_score_col,
        short_score_col=short_score_col,
        long_component_cols=long_component_cols,
        short_component_cols=short_component_cols,
        component_threshold=component_threshold,
        price_col=price_col,
        top_k=int(top_k),
    )


def run_top_k_momentum_baseline(*, panel, price_col, top_k, lookback_days=21, rebalance_freq=None):
    _ = rebalance_freq
    symbols = sorted(panel.index.get_level_values("symbol").unique())
    close = _pivot_rule_panel(panel, price_col, symbols=symbols).replace([np.inf, -np.inf], np.nan).ffill()
    score = close.pct_change(int(lookback_days)).shift(1).replace([np.inf, -np.inf], np.nan)
    common_dates = score.index.intersection(close.index)
    score = score.loc[common_dates]
    close = close.loc[common_dates].fillna(0.0)
    held_side_by_idx = {}
    symbol_to_idx = {sym: idx for idx, sym in enumerate(symbols)}
    position_by_day = pd.DataFrame(0, index=common_dates, columns=symbols, dtype=int)

    for dt in common_dates:
        score_t = score.loc[dt]
        price_ok_t = close.loc[dt].gt(0.0)
        next_held = {}
        for idx, side in sorted(held_side_by_idx.items()):
            if not bool(price_ok_t.iloc[idx]) or (not np.isfinite(score_t.iloc[idx])):
                continue
            current_score = float(score_t.iloc[idx])
            if side > 0 and current_score <= 0.0:
                continue
            if side < 0 and current_score >= 0.0:
                continue
            next_held[idx] = side
        held_side_by_idx = next_held

        capacity = max(0, int(top_k))
        slots_left = max(0, capacity - len(held_side_by_idx))
        if slots_left != 0:
            candidates = []
            for sym in symbols:
                idx = symbol_to_idx[str(sym)]
                if idx in held_side_by_idx or (not bool(price_ok_t.iloc[idx])) or (not np.isfinite(score_t.iloc[idx])):
                    continue
                momentum_value = float(score_t.iloc[idx])
                if momentum_value > 0.0:
                    candidates.append((abs(momentum_value), str(sym), idx, 1))
                elif momentum_value < 0.0:
                    candidates.append((abs(momentum_value), str(sym), idx, -1))
            for _score_value, _sym, idx, side in sorted(candidates, key=lambda row: (row[0], row[1]), reverse=True):
                held_side_by_idx[idx] = int(side)
                if len(held_side_by_idx) >= capacity:
                    break

        for idx, side in sorted(held_side_by_idx.items()):
            position_by_day.loc[dt, symbols[idx]] = int(side)

    return {
        "positions": position_by_day,
        "score": score,
        "close": close,
    }


def resolve_component_cols(score_col):
    mapping = {
        "prob_buy": ["prob_buy"],
        "prob_short": ["prob_short"],
        "buy_score_mean_raw3": ["prob_buy", "pred_rf_reg", "ae_familiarity"],
        "buy_score_mean_raw_pct6": [
            "prob_buy",
            "pred_rf_reg",
            "ae_familiarity",
            "prob_buy_pct",
            "pred_rf_reg_pct",
            "ae_familiarity_pct",
        ],
        "buy_score_pct_mean": ["prob_buy_pct", "pred_rf_reg_pct", "ae_familiarity_pct"],
        "buy_score_pct_product": ["prob_buy_pct", "pred_rf_reg_pct", "ae_familiarity_pct"],
        "buy_score_raw": ["prob_buy", "pred_rf_reg", "ae_familiarity"],
        "buy_score": ["prob_buy", "pred_rf_reg", "ae_familiarity"],
        "short_score_mean_raw3": ["prob_short", "pred_rf_reg", "ae_familiarity"],
        "short_score_mean_raw_pct6": [
            "prob_short",
            "pred_rf_reg",
            "ae_familiarity",
            "prob_short_pct",
            "pred_rf_reg_pct",
            "ae_familiarity_pct",
        ],
        "short_score_pct_mean": ["prob_short_pct", "pred_rf_reg_pct", "ae_familiarity_pct"],
        "short_score_pct_product": ["prob_short_pct", "pred_rf_reg_pct", "ae_familiarity_pct"],
        "short_score_raw": ["prob_short", "pred_rf_reg", "ae_familiarity"],
        "short_score": ["prob_short", "pred_rf_reg", "ae_familiarity"],
        "__momentum_21d__": [],
    }
    return list(mapping.get(str(score_col), [str(score_col)]))


def resolve_short_score_col(score_col):
    mapping = {
        "prob_buy": "prob_short",
        "buy_score_mean_raw3": "short_score_mean_raw3",
        "buy_score_mean_raw_pct6": "short_score_mean_raw_pct6",
        "buy_score_pct_mean": "short_score_pct_mean",
        "buy_score_pct_product": "short_score_pct_product",
        "buy_score_raw": "short_score_raw",
        "buy_score": "short_score",
    }
    key = str(score_col)
    if key in mapping:
        return str(mapping[key])
    if key.startswith("buy_"):
        return "short_" + key[len("buy_"):]
    raise KeyError(f"No short-score mapping configured for: {score_col}")


def _norm_cdf(x):
    x_arr = np.asarray(x, dtype=float)
    if _scipy_ndtr is not None:
        return _scipy_ndtr(x_arr)
    return 0.5 * (1.0 + np.frompyfunc(math.erf, 1, 1)(x_arr / np.sqrt(2.0)).astype(float))


def _norm_cdf_gpu(x_gpu):
    if cp is None:
        raise RuntimeError("CuPy is not available.")
    try:
        from cupyx.scipy.special import ndtr as _cupy_ndtr
        return _cupy_ndtr(x_gpu)
    except Exception:
        return 0.5 * (1.0 + cp.erf(x_gpu / cp.sqrt(2.0)))


def build_realized_vol_panel(close_df, *, window=21, vol_floor=0.15, vol_cap=0.80, annualization=252.0):
    return shared_build_realized_vol_panel(
        close_df,
        window=window,
        vol_floor=vol_floor,
        vol_cap=vol_cap,
        annualization=annualization,
    )


def _black_scholes_panel(close_df, realized_vol_df, *, strike_multiplier, tenor_days=30, rate=0.0, iv_multiplier=1.0, premium_floor=0.25, option_type="call"):
    return shared_build_constant_maturity_call_price_panel(
        close_df,
        realized_vol_df,
        strike_multiplier=strike_multiplier,
        tenor_days=tenor_days,
        rate=rate,
        iv_multiplier=iv_multiplier,
        premium_floor=premium_floor,
    ) if str(option_type).lower() == "call" else shared_build_constant_maturity_put_price_panel(
        close_df,
        realized_vol_df,
        strike_multiplier=strike_multiplier,
        tenor_days=tenor_days,
        rate=rate,
        iv_multiplier=iv_multiplier,
        premium_floor=premium_floor,
    )


def build_constant_maturity_call_price_panel(close_df, realized_vol_df, *, strike_multiplier, tenor_days=30, rate=0.0, iv_multiplier=1.0, premium_floor=0.25):
    return shared_build_constant_maturity_call_price_panel(
        close_df,
        realized_vol_df,
        strike_multiplier=strike_multiplier,
        tenor_days=tenor_days,
        rate=rate,
        iv_multiplier=iv_multiplier,
        premium_floor=premium_floor,
    )


def build_constant_maturity_put_price_panel(close_df, realized_vol_df, *, strike_multiplier, tenor_days=30, rate=0.0, iv_multiplier=1.0, premium_floor=0.25):
    return shared_build_constant_maturity_put_price_panel(
        close_df,
        realized_vol_df,
        strike_multiplier=strike_multiplier,
        tenor_days=tenor_days,
        rate=rate,
        iv_multiplier=iv_multiplier,
        premium_floor=premium_floor,
    )

def backtest_positions_with_directional_asset_returns(
    positions,
    long_asset_returns,
    *,
    short_asset_returns=None,
    initial_balance=100000.0,
    fee_bps=5.0,
    slippage_bps=5.0,
):
    positions = positions.astype(float).copy()
    long_asset_returns = long_asset_returns.reindex(index=positions.index, columns=positions.columns).fillna(0.0)
    if short_asset_returns is None:
        short_asset_returns = -long_asset_returns
    short_asset_returns = short_asset_returns.reindex(index=positions.index, columns=positions.columns).fillna(0.0)
    gross_weight_basis = positions.abs().sum(axis=1).replace(0.0, np.nan)
    weights = positions.div(gross_weight_basis, axis=0).fillna(0.0)
    lagged_weights = weights.shift(1).fillna(0.0)
    long_weights = lagged_weights.clip(lower=0.0)
    short_weights = (-lagged_weights.clip(upper=0.0))
    gross_returns = (long_weights * long_asset_returns + short_weights * short_asset_returns).sum(axis=1)
    turnover = 0.5 * weights.diff().abs().sum(axis=1).fillna(weights.abs().sum(axis=1))
    cost_rate = (float(fee_bps) + float(slippage_bps)) / 10000.0
    net_returns = gross_returns - turnover * cost_rate
    equity = (1.0 + net_returns).cumprod() * float(initial_balance)
    return {
        "weights": weights,
        "turnover": turnover,
        "returns": net_returns,
        "equity": equity,
    }



CUDA available: True | cuML available: True


In [2]:
APP_CFG = {
    "dates": {
        "train_cutoff": "2020-12-31",
        "bt_start": "2021-01-01",
        "bt_end": pd.Timestamp.now(tz="America/Los_Angeles").date().isoformat(),
        "data_start": "2005-01-01",
        "score_start": "2020-01-01",
    },
    "universe": {
        "country": "US",
        "exchanges": list(DEFAULT_US_EXCHANGES),
        "min_market_cap": 10_000_000_000.0,
        "exclude_pooled_vehicles": True,
        "size": None,
    },
    "runtime": {
        "max_clf_train_rows": 250_000_000,
        "label_max_workers": min(8, max(1, (os.cpu_count() or 2) // 2)),
        "fmp_feature_max_workers": min(8, max(1, (os.cpu_count() or 2) // 2)),
        "classifier_backend": "auto",
        "artifact_dir": os.path.abspath("artifacts/synthetic_options_classifier_families"),
    },
    "fmp_refresh": {
        "enabled": True,
        "refresh_symbol_sections_before_build": True,
        "refresh_macro_before_build": False,
        "refresh_classification_performance_before_build": True,
        "refresh_classification_pe_before_build": True,
        "mode": "scoring_ready",
        "skip_cached_inactive_symbols": True,
        "skip_recent_price_attempts": True,
        "existing_historical_sections_only": False,
        "target_start_date": "1900-01-01",
        # Set existing_historical_sections_only=False and early target_start_date
        # so the refresh pulls the maximum possible history for all sections
        # (TTM and non-TTM) instead of only extending existing data.
        "required_historical_sections": [
            "prices_div_adj",
            "key_metrics",
            "ratios",
            "income_statement",
            "income_statement_ttm",
            "income_statement_growth",
            "balance_sheet",
            "balance_sheet_ttm",
            "cash_flow",
            "cash_flow_ttm",
            "cash_flow_growth",
            "financial_growth",
            "earnings",
            "dividends",
            "splits",
        ],
        "max_symbols": None,
        "verbose": True,
    },
    "costs": {
        "fee_bps": 5.0,
        "slippage_bps": 5.0,
    },
    "labels": {
        "k_params": {"YE": list(range(1, 12))},
        "use_sample_weight": False,
        "alpha": 4.0,
        "r_clip": 0.10,
        "horizon_balance": True,
    },
    "probability_columns": {
        "buy_col": "clf__prob_1",
        "short_col": None,
        "infer_short_from_buy": True,
    },
    "strategy": {
        "score_col": "prob_buy",
        "component_threshold": 0.50,
        "top_k_values": [5, 10, 20, 40],
        "strategy_variants": ["classifier_prob", "momentum_21d"],
        "baseline_lookback_days": 21,
    },
    "synthetic_options": {
        "tenor_days": 60,
        "realized_vol_window": 21,
        "vol_floor": 0.15,
        "vol_cap": 0.80,
        "iv_multiplier": 1.0,
        "premium_floor": 0.25,
        "option_buckets": {
            "atm_option": {
                "long_strike_multiplier": 1.00,
                "short_strike_multiplier": 1.00,
            },
            "otm_option": {
                "long_strike_multiplier": 1.05,
                "short_strike_multiplier": 0.95,
            },
            "ditm_option": {
                "long_strike_multiplier": 0.90,
                "short_strike_multiplier": 1.10,
            },
        },
    },
}

display(pd.DataFrame([
    {
        "train_cutoff": APP_CFG["dates"]["train_cutoff"],
        "bt_window": f"{APP_CFG['dates']['bt_start']} to {APP_CFG['dates']['bt_end']}",
        "min_market_cap": APP_CFG["universe"]["min_market_cap"],
        "score_col": APP_CFG["strategy"]["score_col"],
        "component_threshold": APP_CFG["strategy"]["component_threshold"],
        "top_k_values": str(APP_CFG["strategy"]["top_k_values"]),
        "strategy_variants": str(APP_CFG["strategy"]["strategy_variants"]),
        "baseline_lookback_days": APP_CFG["strategy"]["baseline_lookback_days"],
        "option_tenor_days": APP_CFG["synthetic_options"]["tenor_days"],
        "option_buckets": str(APP_CFG["synthetic_options"]["option_buckets"]),
    }
]))


,train_cutoff,bt_window,min_market_cap,score_col,component_threshold,top_k_values,strategy_variants,baseline_lookback_days,option_tenor_days,option_buckets
0,2020-12-31,2021-01-01 to 2026-06-10,1.000000e+10,prob_buy,0.5,"[5, 10, 20, 40]","['classifier_prob', 'momentum_21d']",21,60,"{'atm_option': {'long_strike_multiplier': 1.0,..."


In [3]:
START_DATE = APP_CFG["dates"]["data_start"]
END_DATE = APP_CFG["dates"]["bt_end"]
TRAIN_CUTOFF_TS = pd.Timestamp(APP_CFG["dates"]["train_cutoff"])
BT_START_TS = pd.Timestamp(APP_CFG["dates"]["bt_start"])
BT_END_TS = pd.Timestamp(APP_CFG["dates"]["bt_end"])
SCORE_START_TS = pd.Timestamp(APP_CFG["dates"]["score_start"])

universe_cfg = dict(APP_CFG["universe"])
screener_api_key = str(resolve_fmp_api_key(required=False) or "").strip()
if screener_api_key:
    universe = tuple(
        resolve_symbol_universe_from_screener(
            api_key=screener_api_key,
            marketCapMoreThan=float(universe_cfg["min_market_cap"]),
            country=str(universe_cfg["country"]),
            exchange=",".join(list(universe_cfg["exchanges"])),
            isEtf=False,
            isFund=False,
            isActivelyTrading=None,
            limit=universe_cfg["size"] or 10000,
        )
    )
    universe_source = "FMP screener"
else:
    universe = tuple(
        resolve_symbol_universe(
            min_market_cap=float(universe_cfg["min_market_cap"]),
            country=str(universe_cfg["country"]),
            exchanges=list(universe_cfg["exchanges"]),
            exclude_pooled_vehicles=bool(universe_cfg["exclude_pooled_vehicles"]),
            limit=universe_cfg["size"],
        )
    )
    universe_source = "local DB"
if not universe:
    raise RuntimeError("No symbols resolved for the configured universe.")
log_refresh(f"Resolved {len(universe):,} symbols for the training universe from {universe_source}")
log_memory("resolved universe", rows=len(universe), collect=True)



[05:05:17] Resolved 824 symbols for the training universe from FMP screener
[memory] resolved universe: rss=992.4 MB | rows=824


992.3515625

In [ ]:
fmp_refresh_cfg = dict(APP_CFG.get("fmp_refresh", {}))
fmp_refresh_enabled = bool(fmp_refresh_cfg.get("enabled", False))
fmp_api_key_available = bool(resolve_fmp_api_key(required=False))
with timed_phase("FMP data refresh", rows=len(universe)):
    if fmp_refresh_enabled and fmp_api_key_available:
        log_refresh(
            "FMP data refresh enabled"
            f" | symbols {len(universe):,}"
            f" | mode={str(fmp_refresh_cfg.get('mode') or 'scoring_ready')}"
            f" | sections={tuple(fmp_refresh_cfg.get('required_historical_sections') or ())}"
        )
        run_scoring_data_refresh_from_fmp(
            symbols=universe,
            target_start_date=fmp_refresh_cfg.get("target_start_date") or HISTORICAL_TARGET_MIN_DATE,
            target_end_date=END_DATE,
            refresh_mode=str(fmp_refresh_cfg.get("mode") or "scoring_ready"),
            refresh_symbol_sections_before_build=bool(fmp_refresh_cfg.get("refresh_symbol_sections_before_build", False)),
            refresh_macro_before_build=bool(fmp_refresh_cfg.get("refresh_macro_before_build", False)),
            skip_cached_inactive_symbols=bool(fmp_refresh_cfg.get("skip_cached_inactive_symbols", True)),
            max_symbols=fmp_refresh_cfg.get("max_symbols"),
            existing_historical_sections_only=bool(fmp_refresh_cfg.get("existing_historical_sections_only", False)),
            required_historical_sections=tuple(fmp_refresh_cfg.get("required_historical_sections") or ()),
            verbose=bool(fmp_refresh_cfg.get("verbose", True)),
            progress_logger=log_refresh,
        )
        if bool(fmp_refresh_cfg.get("refresh_classification_performance_before_build", True)):
            try:
                classification_refresh_results = refresh_classification_performance(
                    Symbol.objects.filter(symbol__in=universe).only(
                        "id", "symbol", "exchange", "sector", "industry"
                    ).iterator(),
                    start_date=pd.Timestamp(START_DATE).date(),
                    end_date=pd.Timestamp(END_DATE).date(),
                )
                log_refresh(
                    f"FMP classification performance refresh complete | series={len(classification_refresh_results):,}"
                    f" | observations={sum(int(item['observations_saved']) for item in classification_refresh_results):,}"
                )
            except Exception as exc:
                log_refresh(
                    f"FMP classification performance refresh failed; cached data will be used | {type(exc).__name__}: {exc}"
                )
        if bool(fmp_refresh_cfg.get("refresh_classification_pe_before_build", True)):
            try:
                classification_pe_refresh_results = refresh_classification_pe(
                    Symbol.objects.filter(symbol__in=universe).only(
                        "id", "symbol", "exchange", "sector", "industry"
                    ).iterator(),
                    start_date=pd.Timestamp(START_DATE).date(),
                    end_date=pd.Timestamp(END_DATE).date(),
                )
                log_refresh(
                    f"FMP classification P/E refresh complete | series={len(classification_pe_refresh_results):,}"
                    f" | observations={sum(int(item['observations_saved']) for item in classification_pe_refresh_results):,}"
                )
            except Exception as exc:
                log_refresh(
                    f"FMP classification P/E refresh failed; cached data will be used | {type(exc).__name__}: {exc}"
                )
    else:
        log_refresh(
            "FMP data refresh skipped"
            f" | symbols {len(universe):,}"
            f" | enabled={fmp_refresh_enabled}"
            f" | api_key_available={fmp_api_key_available}"
        )

with timed_phase("technical feature build", rows=len(universe)):
    technical_df, technical_cols, technical_family_frames, technical_family_cols = _build_technical_dataframe_from_django(
        symbols=universe,
        start_date=START_DATE,
        end_date=END_DATE,
    )
if technical_df.empty:
    raise RuntimeError("No technical rows were built from Django prices.")
log_memory("built technical feature panel", rows=len(technical_df), cols=len(technical_df.columns), collect=True)
log_memory("built split pandas-ta-classic technical families", rows=sum(len(frame) for frame in technical_family_frames.values()), cols=sum(len(cols) for cols in technical_family_cols.values()), collect=True)

EXECUTION_PARAMS = {
    "price_col": "close",
    "fee_bps": float(APP_CFG["costs"]["fee_bps"]),
    "slippage_bps": float(APP_CFG["costs"]["slippage_bps"]),
}
WEIGHTING_PARAMS = {
    "use_sample_weight": False,
    "alpha": float(APP_CFG["labels"].get("alpha", 4.0)),
    "r_clip": float(APP_CFG["labels"].get("r_clip", 0.10)),
    "horizon_balance": bool(APP_CFG["labels"].get("horizon_balance", True)),
}

print("Building oracle labels for the full available panel...")
with timed_phase("label dataframe build", rows=len(technical_df)):
    available_price_symbols = set(technical_df.index.get_level_values("symbol"))
    daily_map_all = {
        s: technical_df.xs(s, level="symbol").loc[:BT_END_TS].copy()
        for s in universe
        if s in available_price_symbols
    }
    label_df_all = build_label_dataframe(
        daily_by_symbol=daily_map_all,
        k_params=dict(APP_CFG["labels"]["k_params"]),
        execution_params=EXECUTION_PARAMS,
        weighting=WEIGHTING_PARAMS,
        add_rank_labels=True,
        verbose=False,
        max_workers=int(APP_CFG["runtime"].get("label_max_workers") or 1),
    )
    del daily_map_all
label_df_train = label_df_all.loc[
    label_df_all.index.get_level_values("date") <= TRAIN_CUTOFF_TS
].copy()
label_df_oos = label_df_all.loc[
    (label_df_all.index.get_level_values("date") >= BT_START_TS)
    & (label_df_all.index.get_level_values("date") <= BT_END_TS)
].copy()
log_memory("built full label panel", rows=len(label_df_all), cols=len(label_df_all.columns), collect=True)

scoring_feature_index = technical_df.loc[
    (technical_df.index.get_level_values("date") >= SCORE_START_TS)
    & (technical_df.index.get_level_values("date") <= BT_END_TS)
].index
feature_build_index = label_df_all.index.union(scoring_feature_index).sort_values()
feature_build_index = feature_build_index[feature_build_index.isin(technical_df.index)]
log_memory("resolved model feature build index", rows=len(feature_build_index), collect=True)

# Build one classifier per split pandas-ta-classic technical module, one classifier per selected raw FMP endpoint
# (including each TTM endpoint as its own model),
# separate historical sector/industry performance classifiers, and one macro classifier over economic indicators plus treasury rates.
with timed_phase("FMP endpoint feature-family build", rows=len(universe)):
    fmp_endpoint_frames, fmp_endpoint_cols = shared_build_fmp_endpoint_feature_families(
        symbols=universe,
        target_index=feature_build_index,
        price_panel=technical_df,
        filing_lag_days=45,
        max_workers=int(APP_CFG["runtime"].get("fmp_feature_max_workers") or 1),
        progress_logger=log_refresh,
    )
log_memory("built selected FMP endpoint feature families", rows=sum(len(frame) for frame in fmp_endpoint_frames.values()), cols=sum(len(cols) for cols in fmp_endpoint_cols.values()), collect=True)

with timed_phase("sector and industry performance/P-E feature-family build", rows=len(universe)):
    classification_performance_frames, classification_performance_cols = _build_classification_performance_feature_families(
        symbols=universe,
        target_index=feature_build_index,
        price_panel=technical_df,
        progress_logger=log_refresh,
    )
log_memory(
    "built sector and industry performance/P-E feature families",
    rows=sum(len(frame) for frame in classification_performance_frames.values()),
    cols=sum(len(cols) for cols in classification_performance_cols.values()),
    collect=True,
)

ctx = SimpleNamespace(api_key="")
with timed_phase("macro feature build", rows=len(feature_build_index)):
    macro_df, macro_cols = build_macro_dataframe(
        ctx=ctx,
        start_date=START_DATE,
        end_date=END_DATE,
        target_index=feature_build_index,
        verbose=False,
    )
macro_cols = [c for c in macro_cols if c in macro_df.columns and pd.api.types.is_numeric_dtype(macro_df[c])]
log_memory("built macro economic and treasury feature panel", rows=len(macro_df), cols=len(macro_cols), collect=True)

with timed_phase("classifier feature-frame assembly"):
    technical_model_frames = {
        name: frame.reindex(feature_build_index).loc[:, [c for c in technical_family_cols.get(name, []) if c in frame.columns]]
        for name, frame in technical_family_frames.items()
        if frame is not None and not frame.empty
    }
    if not technical_model_frames:
        technical_model_frames = {
            "prices_div_adj": technical_df.reindex(feature_build_index).loc[:, [c for c in technical_cols if c in technical_df.columns]]
        }
    feature_family_frames = {
        **technical_model_frames,
        **{
            name: frame
            for name, frame in fmp_endpoint_frames.items()
            if frame is not None and not frame.empty and frame.shape[1] > 0
        },
        **{
            name: frame
            for name, frame in classification_performance_frames.items()
            if frame is not None and not frame.empty and frame.shape[1] > 0
        },
        **({"macro_economic_treasury": macro_df.loc[:, macro_cols]} if macro_cols else {}),
    }
    feature_family_frames = {
        name: frame
        for name, frame in feature_family_frames.items()
        if frame is not None and not frame.empty and frame.shape[1] > 0
    }
    required_classification_families = {
        "sector_performance", "industry_performance", "sector_pe", "industry_pe"
    }
    missing_classification_families = sorted(required_classification_families - set(feature_family_frames))
    if missing_classification_families:
        raise RuntimeError(
            "Missing required FMP classification performance families: "
            + ", ".join(missing_classification_families)
            + ". Refresh historical sector/industry performance and P/E before training the MoE."
        )
    feature_family_frames = {
        name: frame.loc[~frame.index.duplicated(keep="last")].copy() if not frame.index.is_unique else frame
        for name, frame in feature_family_frames.items()
    }
    feature_family_weights = {name: 1.0 for name in feature_family_frames}
    del fmp_endpoint_frames, classification_performance_frames
    gc.collect()
log_memory(
    "built selected classifier feature frames",
    rows=sum(len(frame) for frame in feature_family_frames.values()),
    cols=sum(frame.shape[1] for frame in feature_family_frames.values()),
    collect=True,
)


[timer] start FMP data refresh | rss=992.4 MB
[05:05:17] FMP data refresh enabled | symbols 824 | mode=scoring_ready | sections=('prices_div_adj', 'key_metrics', 'ratios', 'income_statement', 'income_statement_ttm', 'income_statement_growth', 'balance_sheet', 'balance_sheet_ttm', 'cash_flow', 'cash_flow_ttm', 'cash_flow_growth', 'financial_growth', 'earnings', 'dividends', 'splits')
[05:05:17] Refreshing latest price data from FMP before feature build | total symbols 824 | targeted symbols 0 | needs refresh 0 | already fresh 824
[05:05:17] FMP price refresh complete | refreshed 0 | errors 0
[05:05:19] Refreshing FMP section key_metrics before feature build | total symbols 824 | targeted symbols 7 | needs refresh 7 | already fresh 817
[05:05:19] Top key_metrics refresh reasons | key_metrics:full=7
[05:05:19] FMP section key_metrics | FMP historical refresh start [1/7] DTY
[05:05:21] FMP section key_metrics | FMP historical refresh done  [1/7] DTY | status=ok | reason=key_metrics:full | 

In [ ]:
if BT_START_TS <= TRAIN_CUTOFF_TS:
    raise ValueError(
        "Backtest start must be after train_cutoff so train/OOS labels do not overlap: "
        f"bt_start={BT_START_TS.date()} train_cutoff={TRAIN_CUTOFF_TS.date()}"
    )

print("Label/date split")
display(pd.DataFrame([
    {
        "train_cutoff": TRAIN_CUTOFF_TS.date().isoformat(),
        "bt_window": f"{BT_START_TS.date()} to {BT_END_TS.date()}",
        "all_label_rows": len(label_df_all),
        "train_label_rows": len(label_df_train),
        "oos_label_rows": len(label_df_oos),
        "feature_family_count": len(feature_family_frames),
        "feature_families": ", ".join(feature_family_frames.keys()),
        "feature_cols_by_family": str({name: int(frame.shape[1]) for name, frame in feature_family_frames.items()}),
        "classifier_family_weights": str(feature_family_weights),
    }
]))


def _cap_training_rows(df, max_rows, *, random_state=1337):
    max_rows = int(max_rows or 0)
    if max_rows <= 0 or len(df) <= max_rows:
        return df
    return df.sample(
        n=max_rows,
        random_state=random_state,
        replace=False,
    ).sort_index()


def _sanitize_family_numeric_features(df, feature_cols=None, *, clip_abs=1e12):
    if df is None or df.empty:
        return pd.DataFrame(), []
    out = df.copy()
    cols = list(feature_cols or out.columns)
    cols = [c for c in cols if c in out.columns and pd.api.types.is_numeric_dtype(out[c])]
    if not cols:
        return out.iloc[0:0].copy(), []
    out.loc[:, cols] = out[cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    if clip_abs is not None and float(clip_abs) > 0.0:
        out.loc[:, cols] = out[cols].clip(lower=-float(clip_abs), upper=float(clip_abs))
    cols = [c for c in cols if out[c].notna().any()]
    if not cols:
        return out.iloc[0:0].copy(), []
    available_mask = out[cols].notna().any(axis=1)
    return out.loc[available_mask].copy(), cols


def _filter_family_available_rows(df, feature_cols=None):
    out, _cols = _sanitize_family_numeric_features(df, feature_cols)
    return out


def _downcast_numeric_frame(df, feature_cols=None):
    if df is None or df.empty:
        return df
    cols = list(feature_cols or df.columns)
    cols = [c for c in cols if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
    if not cols:
        return df
    out = df.copy()
    out.loc[:, cols] = out[cols].astype("float32")
    return out


def _dedupe_panel_index(df, *, keep="last"):
    if df is None or df.empty or df.index.is_unique:
        return df
    return df.loc[~df.index.duplicated(keep=keep)].copy()


def _build_feature_slice_from_families(*, start_ts=None, end_ts=None, cols_by_family=None):
    frames = []
    for family_name, family_df in feature_family_frames.items():
        frame = family_df
        if start_ts is not None:
            frame = frame.loc[frame.index.get_level_values("date") >= start_ts]
        if end_ts is not None:
            frame = frame.loc[frame.index.get_level_values("date") <= end_ts]
        if frame.empty:
            continue
        cols = list((cols_by_family or feature_list_by_family).get(family_name, []))
        cols = [col for col in cols if col in frame.columns]
        if not cols:
            continue
        frames.append(_dedupe_panel_index(frame[cols]))
    if not frames:
        return pd.DataFrame()
    out = pd.concat(frames, axis=1).sort_index()
    if out.columns.has_duplicates:
        out = out.loc[:, ~out.columns.duplicated(keep="last")]
    return out


def _build_feature_slice_for_index(row_index, *, cols_by_family=None):
    if row_index is None or len(row_index) == 0:
        return pd.DataFrame(index=row_index)
    frames = []
    family_cols = cols_by_family or feature_list_by_family
    for family_name, family_df in feature_family_frames.items():
        cols = list(family_cols.get(family_name, []))
        cols = [col for col in cols if col in family_df.columns]
        if not cols:
            continue
        family_df = _dedupe_panel_index(family_df)
        frames.append(family_df.reindex(row_index)[cols])
    if not frames:
        return pd.DataFrame(index=row_index)
    out = pd.concat(frames, axis=1)
    if out.columns.has_duplicates:
        out = out.loc[:, ~out.columns.duplicated(keep="last")]
    return out


def _family_coverage_summary(df, feature_cols=None):
    available = _filter_family_available_rows(df, feature_cols)
    if available.empty:
        return {"coverage_start": None, "coverage_end": None, "available_rows": 0}
    dates = pd.DatetimeIndex(pd.to_datetime(available.index.get_level_values("date")))
    return {
        "coverage_start": dates.min().date().isoformat(),
        "coverage_end": dates.max().date().isoformat(),
        "available_rows": int(len(available)),
    }


family_models = {}
family_training_rows = []
all_family_feature_cols = []

def _resolve_classifier_backend():
    requested = str(APP_CFG["runtime"].get("classifier_backend") or "auto").strip().lower()
    if requested in {"auto", "cuda", "gpu"}:
        return "cuml_rf" if CUML_AVAILABLE else "sklearn_rf"
    return requested


def _make_classifier_model():
    backend = _resolve_classifier_backend()
    common_kwargs = {
        "random_state": 1337,
        "n_estimators": 200,
        "max_depth": 12,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
    }
    if backend == "cuml_rf":
        return CumlRFClassifier(**common_kwargs)
    if backend == "sklearn_rf":
        return SklearnRFClassifier(**common_kwargs, class_weight="balanced", n_jobs=-1)
    raise ValueError(f"Unsupported classifier_backend: {backend}")


classifier_backend = _resolve_classifier_backend()
print(f"Classifier backend: {classifier_backend} | requested={APP_CFG['runtime'].get('classifier_backend')} | cuda_available={CUDA_AVAILABLE} | cuml_available={CUML_AVAILABLE}")

for family_name, family_df in feature_family_frames.items():
    if family_df.empty or family_df.shape[1] == 0:
        print(f"Skipping {family_name}: no feature columns.")
        continue

    with timed_phase(f"train classifier family {family_name}", rows=len(family_df), cols=family_df.shape[1]):
        family_coverage = _family_coverage_summary(family_df)
        family_features_train = family_df.loc[
            family_df.index.get_level_values("date") <= TRAIN_CUTOFF_TS
        ].copy()
        family_features_train = _filter_family_available_rows(family_features_train)
        if family_features_train.empty:
            print(f"Skipping {family_name}: no endpoint data available before train_cutoff.")
            continue

        family_train_df, family_feature_cols, _ = prepare_ml_dataset(
            features_df=family_features_train,
            labels_df=label_df_train,
            target_cols=["target", "trade_return"],
            weight_col=None,
            config=MLDatasetConfig(drop_nan_features=False),
            verbose=True,
        )
        if family_train_df.empty or not family_feature_cols:
            print(f"Skipping {family_name}: no train rows or active features after label join.")
            continue
        family_train_df, family_feature_cols = _sanitize_family_numeric_features(family_train_df, family_feature_cols)
        family_train_df = _downcast_numeric_frame(family_train_df, family_feature_cols)
        if family_train_df.empty or not family_feature_cols:
            print(f"Skipping {family_name}: no train rows with finite endpoint features after label join.")
            continue

        family_clf_train_df = _cap_training_rows(
            family_train_df,
            APP_CFG["runtime"]["max_clf_train_rows"],
            random_state=1337,
        )
        family_clf_train_df, family_feature_cols = _sanitize_family_numeric_features(family_clf_train_df, family_feature_cols)
        family_clf_train_df = _downcast_numeric_frame(family_clf_train_df, family_feature_cols)
        if family_clf_train_df.empty or not family_feature_cols:
            print(f"Skipping {family_name}: no finite rows remained after training cap.")
            continue

        log_memory(f"prepared {family_name} RF training data", rows=len(family_clf_train_df), cols=len(family_feature_cols), collect=True)
        print(f"Training {family_name} classifier with {len(family_feature_cols)} features...")
        spec_clf = FitSpec(
            feature_cols=list(family_feature_cols),
            target_col="target",
            weight_col=None,
            split_ratio=1.0,
            model_tag=f"classifier ({family_name}): predicts buy/positive class from target",
        )
        family_clf = _make_classifier_model()
        family_clf.fit(family_clf_train_df, spec_clf, verbose=True)
        log_memory(f"fit {family_name} RF classifier", rows=len(family_clf_train_df), cols=len(family_feature_cols), collect=True)

        family_train_rows = int(len(family_train_df))
        family_clf_fit_rows = int(len(family_clf_train_df))
        family_models[family_name] = {
            "feature_df": family_df,
            "feature_cols": list(family_feature_cols),
            "train_rows": family_train_rows,
            "clf_fit_rows": family_clf_fit_rows,
            "clf": family_clf,
            "coverage": family_coverage,
        }
        all_family_feature_cols.extend(str(c) for c in family_feature_cols)
        family_training_rows.append(
            {
                "family": family_name,
                "feature_cols": len(family_feature_cols),
                "train_rows": family_train_rows,
                "clf_fit_rows": family_clf_fit_rows,
                "available_rows_full_panel": int(family_coverage["available_rows"]),
                "coverage_start": family_coverage["coverage_start"],
                "coverage_end": family_coverage["coverage_end"],
                "weight": float(feature_family_weights.get(family_name, 1.0)),
            }
        )
        del family_features_train, family_train_df, family_clf_train_df
        gc.collect()

if not family_models:
    raise RuntimeError("No classifier models were trained.")
missing_trained_classification_models = sorted(
    {"sector_performance", "industry_performance", "sector_pe", "industry_pe"} - set(family_models)
)
if missing_trained_classification_models:
    raise RuntimeError(
        "Required classification-performance MoE models did not train: "
        + ", ".join(missing_trained_classification_models)
    )

family_training_df = pd.DataFrame(family_training_rows)
print("Per-FMP-family classifier training summary")
display(family_training_df)

clf_raw = next(iter(family_models.values()))["clf"]
raw_feature_list = sorted(set(all_family_feature_cols))
model_source = "retrained_per_fmp_family_classifiers"
artifact_status = {"age": pd.NaT, "reason": "forced_retrain"}

feature_list_by_family = {name: list(payload["feature_cols"]) for name, payload in family_models.items()}

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

INSAMPLE_START_TS = pd.Timestamp(START_DATE)
INSAMPLE_END_TS = TRAIN_CUTOFF_TS
OUTSAMPLE_START_TS = BT_START_TS
OUTSAMPLE_END_TS = BT_END_TS

print("Building aligned classifier diagnostics from fitted family models...")
print(f"  - In-sample:     {INSAMPLE_START_TS.date()} to {INSAMPLE_END_TS.date()}")
print(f"  - Out-of-sample: {OUTSAMPLE_START_TS.date()} to {OUTSAMPLE_END_TS.date()}")


def _positive_class_index(model, proba):
    if hasattr(model, "positive_class_index"):
        idx = int(model.positive_class_index())
        return idx if 0 <= idx < proba.shape[1] else proba.shape[1] - 1
    raw_model = getattr(model, "model", model)
    classes = list(getattr(raw_model, "classes_", []))
    if 1 in classes:
        return classes.index(1)
    if "1" in classes:
        return classes.index("1")
    return proba.shape[1] - 1


def _predict_rf_classifier_probability(model, features, used_features):
    if hasattr(model, "predict_proba"):
        proba = np.asarray(model.predict_proba(features, feature_cols=used_features), dtype=float)
    else:
        raw_model = getattr(model, "model", model)
        proba = np.asarray(raw_model.predict_proba(features[used_features]), dtype=float)
    return proba[:, _positive_class_index(model, proba)]


def _score_family_predictions(feature_df, family_payload, *, start_ts, end_ts, family_name, symbol=None, row_mask=None, progress_logger=None):
    phase_start = time.perf_counter()
    used_features = list(getattr(family_payload["clf"], "_used_features", []))
    if not used_features:
        raise RuntimeError(f"Classifier for {family_name} does not expose fitted _used_features.")
    used_features = [col for col in used_features if col in feature_df.columns]
    if not used_features:
        return pd.DataFrame()

    if row_mask is None:
        date_values = feature_df.index.get_level_values("date")
        row_mask = (date_values >= start_ts) & (date_values <= end_ts)
        if symbol is not None:
            symbol_values = feature_df.index.get_level_values("symbol").astype(str).str.upper()
            row_mask = row_mask & (symbol_values == str(symbol).strip().upper())
    features = feature_df.loc[row_mask, used_features]
    if callable(progress_logger):
        progress_logger(
            f"Scoring prepare done | family={family_name} | candidate rows={len(features):,}"
            f" | features={len(used_features):,} | elapsed={time.perf_counter() - phase_start:.2f}s"
        )
    if features.empty:
        return pd.DataFrame(index=features.index)

    matrix_start = time.perf_counter()
    x = features.to_numpy(dtype=np.float32, copy=False)
    finite_mask = np.isfinite(x).any(axis=1)
    if not bool(finite_mask.any()):
        if callable(progress_logger):
            progress_logger(f"Scoring skipped | family={family_name} | no finite feature rows")
        return pd.DataFrame(index=features.index[:0])
    dropped_rows = 0
    if not bool(finite_mask.all()):
        dropped_rows = int((~finite_mask).sum())
        features = features.iloc[np.flatnonzero(finite_mask)]
        x = x[finite_mask]

    # Build a compact float32 frame so cuML avoids repeated dtype conversion/copy work.
    score_features = pd.DataFrame(x, index=features.index, columns=used_features, copy=False)
    if callable(progress_logger):
        progress_logger(
            f"Scoring predict start | family={family_name} | rows={len(score_features):,}"
            f" | features={len(used_features):,} | dropped_nonfinite={dropped_rows:,}"
            f" | matrix_elapsed={time.perf_counter() - matrix_start:.2f}s"
        )
    predict_start = time.perf_counter()
    prob_buy = _predict_rf_classifier_probability(family_payload["clf"], score_features, used_features)
    predict_elapsed = time.perf_counter() - predict_start
    if callable(progress_logger):
        progress_logger(
            f"Scoring predict done  | family={family_name} | rows={len(score_features):,}"
            f" | predict_elapsed={predict_elapsed:.2f}s | total_elapsed={time.perf_counter() - phase_start:.2f}s"
        )
    out = pd.DataFrame(index=score_features.index)
    out[f"{family_name}__prob_buy"] = prob_buy
    return _dedupe_panel_index(out)


def _build_family_prediction_panel(*, start_ts, end_ts, progress_logger=None, progress_every=25):
    frames = []
    active_weights = {}
    for family_name, payload in family_models.items():
        family_start = time.perf_counter()
        feature_df = payload["feature_df"]
        date_values = feature_df.index.get_level_values("date")
        symbol_values = feature_df.index.get_level_values("symbol").astype(str).str.upper()
        window_mask = (date_values >= start_ts) & (date_values <= end_ts)
        score_symbols = tuple(pd.Index(symbol_values[window_mask]).unique().sort_values().tolist())
        total_symbols = len(score_symbols)
        candidate_rows = int(window_mask.sum())
        if callable(progress_logger):
            progress_logger(
                f"Scoring family start | family={family_name} | symbols={total_symbols:,}"
                f" | candidate rows={candidate_rows:,} | window={start_ts.date()} to {end_ts.date()}"
            )

        scored = _score_family_predictions(
            feature_df,
            payload,
            start_ts=start_ts,
            end_ts=end_ts,
            family_name=family_name,
            row_mask=window_mask,
            progress_logger=progress_logger,
        )
        if scored.empty:
            if callable(progress_logger):
                progress_logger(
                    f"Scoring family complete | family={family_name} | status=empty"
                    f" | elapsed={time.perf_counter() - family_start:.2f}s"
                )
            continue

        frames.append(_dedupe_panel_index(scored.sort_index()))
        active_weights[family_name] = float(feature_family_weights.get(family_name, 1.0))
        if callable(progress_logger):
            scored_symbol_values = scored.index.get_level_values("symbol").astype(str).str.upper()
            row_counts = pd.Series(scored_symbol_values).value_counts().reindex(score_symbols, fill_value=0)
            scored_symbols = int((row_counts > 0).sum())
            scored_rows = int(row_counts.sum())
            min_rows = int(row_counts[row_counts > 0].min()) if scored_symbols else 0
            max_rows = int(row_counts.max()) if len(row_counts) else 0
            empty_symbols = int((row_counts == 0).sum())
            progress_logger(
                f"Scoring family complete | family={family_name} | scored symbols={scored_symbols:,}/{total_symbols:,}"
                f" | rows={scored_rows:,} | empty symbols={empty_symbols:,}"
                f" | rows/symbol min={min_rows:,} max={max_rows:,}"
                f" | elapsed={time.perf_counter() - family_start:.2f}s"
            )
    if not frames:
        return pd.DataFrame()

    out = pd.concat(frames, axis=1).sort_index()
    weight_sum = sum(active_weights.values())
    if weight_sum <= 0.0:
        raise RuntimeError("Classifier family weights must sum to a positive value.")

    weighted_sum = pd.Series(0.0, index=out.index, dtype=float)
    row_weight_sum = pd.Series(0.0, index=out.index, dtype=float)
    row_family_count = pd.Series(0, index=out.index, dtype=int)
    for family_name, weight in active_weights.items():
        prob = pd.to_numeric(out[f"{family_name}__prob_buy"], errors="coerce")
        valid = prob.notna()
        weighted_sum.loc[valid] += float(weight) * prob.loc[valid]
        row_weight_sum.loc[valid] += float(weight)
        row_family_count.loc[valid] += 1
    out["clf__prob_1"] = (weighted_sum / row_weight_sum.replace(0.0, np.nan)).clip(0.0, 1.0)
    out["clf"] = out["clf__prob_1"]
    out["classifier_available_families"] = row_family_count
    out["classifier_active_families"] = len(active_weights)
    # Compatibility columns for enrich_scored_panel. These are neutral classifier-only values.
    out["ranking"] = out["clf__prob_1"]
    out["ae_familiarity"] = 1.0
    log_memory(f"scored classifier family slice {start_ts.date()} to {end_ts.date()}", rows=len(out), cols=len(out.columns), collect=True)
    return out


def _build_single_family_prediction_panel(*, family_name, start_ts, end_ts, progress_logger=None):
    payload = family_models.get(family_name)
    if payload is None:
        raise KeyError(f"Unknown family: {family_name}")

    scored = _score_family_predictions(
        payload["feature_df"],
        payload,
        start_ts=start_ts,
        end_ts=end_ts,
        family_name=family_name,
        progress_logger=progress_logger,
    )
    if scored.empty:
        return scored

    out = _dedupe_panel_index(scored.sort_index())
    prob_col = f"{family_name}__prob_buy"
    out["clf__prob_1"] = pd.to_numeric(out[prob_col], errors="coerce").fillna(0.0)
    out["clf"] = out["clf__prob_1"]
    out["ranking"] = out["clf__prob_1"]
    out["ae_familiarity"] = 1.0
    out["classifier_available_families"] = 1
    out["classifier_active_families"] = 1
    return out


def _build_direct_model_diagnostic_frame(split_name, labels_df, *, start_ts, end_ts):
    scored = _build_family_prediction_panel(start_ts=start_ts, end_ts=end_ts, progress_logger=log_refresh)
    labels = labels_df.loc[
        (labels_df.index.get_level_values("date") >= start_ts)
        & (labels_df.index.get_level_values("date") <= end_ts)
    ].copy()
    diagnostic_label_cols = [col for col in ["target", "trade_return"] if col in labels.columns]
    out = scored.join(labels[diagnostic_label_cols], how="inner")
    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.dropna(subset=["clf__prob_1"])
    if out.empty:
        return out
    out["split"] = str(split_name)
    out["window_start"] = start_ts.date().isoformat()
    out["window_end"] = end_ts.date().isoformat()
    out["target_binary"] = (pd.to_numeric(out["target"], errors="coerce") > 0).astype(int)
    out["pred_binary"] = (pd.to_numeric(out["clf__prob_1"], errors="coerce") >= 0.50).astype(int)
    out["prob_buy"] = pd.to_numeric(out["clf__prob_1"], errors="coerce").fillna(0.0)
    out["diagnostic_year"] = out.index.get_level_values("date").year
    return out


label_df_insample_diagnostic = label_df_train.loc[
    (label_df_train.index.get_level_values("date") >= INSAMPLE_START_TS)
    & (label_df_train.index.get_level_values("date") <= INSAMPLE_END_TS)
].copy()
label_df_outsample_diagnostic = label_df_oos.loc[
    (label_df_oos.index.get_level_values("date") >= OUTSAMPLE_START_TS)
    & (label_df_oos.index.get_level_values("date") <= OUTSAMPLE_END_TS)
].copy()

model_diagnostic_frames = [
    _build_direct_model_diagnostic_frame(
        "in_sample_train_window",
        label_df_insample_diagnostic,
        start_ts=INSAMPLE_START_TS,
        end_ts=INSAMPLE_END_TS,
    ),
    _build_direct_model_diagnostic_frame(
        "out_of_sample_backtest_window",
        label_df_outsample_diagnostic,
        start_ts=OUTSAMPLE_START_TS,
        end_ts=OUTSAMPLE_END_TS,
    ),
]
model_diagnostic_frames = [frame for frame in model_diagnostic_frames if not frame.empty]
model_diagnostic_df = pd.concat(model_diagnostic_frames, axis=0).sort_index() if model_diagnostic_frames else pd.DataFrame()

if model_diagnostic_df.empty:
    raise RuntimeError("No rows with both classifier scores and realized labels were available for aligned diagnostics.")


def _safe_roc_auc(labels, scores):
    return float(roc_auc_score(labels, scores)) if pd.Series(labels).nunique(dropna=True) == 2 else np.nan


def _summarize_model_split(g):
    y_true = g["target_binary"].to_numpy(dtype=int)
    y_pred = g["pred_binary"].to_numpy(dtype=int)
    y_prob = g["prob_buy"].to_numpy(dtype=float)
    return pd.Series({
        "window": f"{g['window_start'].iloc[0]} to {g['window_end'].iloc[0]}",
        "rows": int(len(g)),
        "symbols": int(g.index.get_level_values("symbol").nunique()),
        "dates": int(g.index.get_level_values("date").nunique()),
        "positive_rate_actual": float(g["target_binary"].mean()),
        "positive_rate_predicted": float(g["pred_binary"].mean()),
        "accuracy_at_50pct": float(accuracy_score(y_true, y_pred)),
        "roc_auc_prob_buy": _safe_roc_auc(y_true, y_prob),
        "avg_actual_trade_return": float(pd.to_numeric(g["trade_return"], errors="coerce").mean()),
    })


split_model_diagnostics = (
    model_diagnostic_df
    .groupby("split", sort=False)
    .apply(_summarize_model_split)
    .reset_index()
)

print("Aligned classifier diagnostics: train_cutoff vs backtest window")
display(split_model_diagnostics)

family_oos_diagnostics = []
for family_name in family_models:
    family_prob_col = f"{family_name}__prob_buy"
    for split_name, split_df in model_diagnostic_df.groupby("split", sort=False):
        if family_prob_col not in split_df.columns:
            continue
        y_true = split_df["target_binary"].to_numpy(dtype=int)
        y_prob = pd.to_numeric(split_df[family_prob_col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        y_pred = (y_prob >= 0.50).astype(int)
        family_oos_diagnostics.append({
            "split": split_name,
            "family": family_name,
            "rows": int(len(split_df)),
            "accuracy_at_50pct": float(accuracy_score(y_true, y_pred)),
            "roc_auc_prob_buy": _safe_roc_auc(y_true, y_prob),
        })

print("Classifier family diagnostics")
display(pd.DataFrame(family_oos_diagnostics))

for split_name, split_df in model_diagnostic_df.groupby("split", sort=False):
    print(f"Classification report: classifier families {split_name} | positive class = realized profitable/buy label")
    print(classification_report(
        split_df["target_binary"],
        split_df["pred_binary"],
        labels=[0, 1],
        target_names=["not_buy", "buy"],
        zero_division=0,
    ))
    print(f"Confusion matrix: classifier families {split_name}")
    display(pd.DataFrame(
        confusion_matrix(split_df["target_binary"], split_df["pred_binary"], labels=[0, 1]),
        index=["actual_not_buy", "actual_buy"],
        columns=["pred_not_buy", "pred_buy"],
    ))

yearly_model_diagnostics = (
    model_diagnostic_df
    .groupby(["split", "diagnostic_year"], sort=False)
    .apply(lambda g: pd.Series({
        "rows": int(len(g)),
        "positive_rate_actual": float(g["target_binary"].mean()),
        "positive_rate_predicted": float(g["pred_binary"].mean()),
        "accuracy_at_50pct": float(accuracy_score(g["target_binary"], g["pred_binary"])),
        "roc_auc_prob_buy": _safe_roc_auc(g["target_binary"], g["prob_buy"]),
        "avg_actual_trade_return": float(pd.to_numeric(g["trade_return"], errors="coerce").mean()),
    }))
    .reset_index()
)

print("Yearly classifier diagnostics by aligned split")
display(yearly_model_diagnostics)

score_decile_diagnostics = []
for split_name, split_df in model_diagnostic_df.groupby("split", sort=False):
    decile_df = split_df.copy()
    q = min(10, max(1, int(decile_df["prob_buy"].notna().sum())))
    decile_df["prob_buy_decile"] = pd.qcut(
        decile_df["prob_buy"].rank(method="first"),
        q=q,
        labels=False,
        duplicates="drop",
    )
    decile_summary = (
        decile_df
        .groupby("prob_buy_decile", dropna=True)
        .agg(
            rows=("target_binary", "size"),
            mean_prob_buy=("prob_buy", "mean"),
            hit_rate=("target_binary", "mean"),
            avg_actual_trade_return=("trade_return", "mean"),
        )
        .reset_index()
    )
    decile_summary.insert(0, "split", split_name)
    score_decile_diagnostics.append(decile_summary)

score_decile_diagnostics = pd.concat(score_decile_diagnostics, ignore_index=True) if score_decile_diagnostics else pd.DataFrame()

print("Probability decile diagnostics by aligned split")
display(score_decile_diagnostics)

feature_importance_rows = []
for family_name, payload in family_models.items():
    for feature, importance in sorted(payload["clf"].feature_importance().items(), key=lambda item: item[1], reverse=True)[:15]:
        feature_importance_rows.append({
            "family": family_name,
            "model": "classifier",
            "feature": feature,
            "importance": float(importance),
        })
feature_importance_df = pd.DataFrame(feature_importance_rows)
print("Top classifier feature importances by FMP family")
display(feature_importance_df)

artifact_dir = Path(str(APP_CFG["runtime"]["artifact_dir"]))
artifact_dir.mkdir(parents=True, exist_ok=True)
with open(artifact_dir / "classifier_families.pkl", "wb") as f:
    pickle.dump({name: payload["clf"] for name, payload in family_models.items()}, f)
with open(artifact_dir / "classifier_families_meta.json", "w") as f:
    json.dump(
        {
            "artifact_version": 1,
            "stack": "feature_family_moe_with_sector_industry_performance_and_pe",
            "trained_families": list(family_models.keys()),
            "feature_family_weights": {name: float(feature_family_weights.get(name, 1.0)) for name in family_models},
            "feature_list_by_family": {name: list(payload["feature_cols"]) for name, payload in family_models.items()},
            "classifier_only": True,
        },
        f,
        indent=2,
    )
saved_artifact_dir = artifact_dir

display(pd.DataFrame([
    {
        "universe_size": len(universe),
        "model_feature_source": "technical + FMP endpoint + sector/industry performance + sector/industry P/E + macro classifiers",
        "trained_families": ", ".join(family_models.keys()),
        "feature_panel_rows": max(len(payload["feature_df"]) for payload in family_models.values()),
        "feature_panel_cols": sum(len(payload["feature_cols"]) for payload in family_models.values()),
        "feature_family_count": len(family_models),
        "feature_cols_by_family": str({name: len(payload["feature_cols"]) for name, payload in family_models.items()}),
        "train_rows_total_across_families": int(family_training_df["train_rows"].sum()),
        "clf_fit_rows_total_across_families": int(family_training_df["clf_fit_rows"].sum()),
        "artifact_dir": str(saved_artifact_dir),
        "model_source": model_source,
        "classifier_backend": classifier_backend,
        "artifact_age_hours": None if pd.isna(artifact_status.get("age")) else round(float(artifact_status["age"] / pd.Timedelta(hours=1)), 2),
        "artifact_status": str(artifact_status.get("reason") or ""),
    }
]))


In [ ]:
prob_cfg = ProbabilityColumnConfig(**APP_CFG["probability_columns"])
with timed_phase("final classifier scoring", collect_after=True):
    scored_panel_all = _build_family_prediction_panel(
        start_ts=SCORE_START_TS,
        end_ts=BT_END_TS,
        progress_logger=log_refresh,
    )
if scored_panel_all.empty:
    raise RuntimeError("No ensemble score rows were produced for the scoring panel.")

with timed_phase("enrich scored panel", rows=len(scored_panel_all), cols=len(scored_panel_all.columns)):
    scored_panel_all = enrich_scored_panel(scored_panel_all, prob_config=prob_cfg)
with timed_phase("make backtest panel", rows=len(scored_panel_all), cols=len(scored_panel_all.columns)):
    bt_panel_all = make_backtest_panel(
        scored_panel=scored_panel_all,
        technical_df=technical_df,
        start=SCORE_START_TS,
        end=BT_END_TS,
    )
with timed_phase("slice OOS backtest panel", rows=len(bt_panel_all), cols=len(bt_panel_all.columns)):
    bt_panel_5y = bt_panel_all.loc[
        (bt_panel_all.index.get_level_values("date") >= BT_START_TS)
        & (bt_panel_all.index.get_level_values("date") <= BT_END_TS)
    ].copy()

display(pd.DataFrame([
    {
        "scoring_rows": len(bt_panel_all),
        "oos_rows": len(bt_panel_5y),
        "oos_symbols": int(bt_panel_5y.index.get_level_values("symbol").nunique()),
        "oos_dates": int(bt_panel_5y.index.get_level_values("date").nunique()),
        "trained_families": ", ".join(family_models.keys()),
        "classifier_family_weights": str(feature_family_weights),
        "score_source": "feature-family MoE probabilities including sector/industry performance and P/E",
    }
]))


In [ ]:
score_col = str(APP_CFG["strategy"]["score_col"])
short_score_col = resolve_short_score_col(score_col)
component_cols = resolve_component_cols(score_col)
short_component_cols = resolve_component_cols(short_score_col)
top_k_values = list(APP_CFG["strategy"]["top_k_values"])
strategy_variants = list(APP_CFG["strategy"].get("strategy_variants", ["classifier_prob"]))
years = list(range(BT_START_TS.year, BT_END_TS.year + 1))

with timed_phase("prepare rule input panels", rows=len(bt_panel_5y), cols=len(bt_panel_5y.columns)):
    base_inputs = _prepare_capacity_rule_inputs(bt_panel_5y, score_col, component_cols, "close")
    close_panel = base_inputs["close"]
with timed_phase("build realized volatility panel", rows=close_panel.size):
    realized_vol_panel = build_realized_vol_panel(
        close_panel,
        window=int(APP_CFG["synthetic_options"]["realized_vol_window"]),
        vol_floor=float(APP_CFG["synthetic_options"]["vol_floor"]),
        vol_cap=float(APP_CFG["synthetic_options"]["vol_cap"]),
    )

with timed_phase("build instrument return panels", rows=close_panel.size):
    instrument_return_panels = {
        "equity": {
            "long": close_panel.pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0),
            "short": close_panel.pct_change().mul(-1.0).replace([np.inf, -np.inf], np.nan).fillna(0.0),
        },
    }
    synthetic_price_panels = {}
    for option_name, option_bucket in APP_CFG["synthetic_options"]["option_buckets"].items():
        with timed_phase(f"build synthetic option panel {option_name}", rows=close_panel.size):
            long_strike_multiplier = float(option_bucket["long_strike_multiplier"])
            short_strike_multiplier = float(option_bucket["short_strike_multiplier"])
            call_price_panel = build_constant_maturity_call_price_panel(
                close_panel,
                realized_vol_panel,
                strike_multiplier=long_strike_multiplier,
                tenor_days=int(APP_CFG["synthetic_options"]["tenor_days"]),
                iv_multiplier=float(APP_CFG["synthetic_options"]["iv_multiplier"]),
                premium_floor=float(APP_CFG["synthetic_options"]["premium_floor"]),
            )
            put_price_panel = build_constant_maturity_put_price_panel(
                close_panel,
                realized_vol_panel,
                strike_multiplier=short_strike_multiplier,
                tenor_days=int(APP_CFG["synthetic_options"]["tenor_days"]),
                iv_multiplier=float(APP_CFG["synthetic_options"]["iv_multiplier"]),
                premium_floor=float(APP_CFG["synthetic_options"]["premium_floor"]),
            )
            synthetic_price_panels[option_name] = {
                "call": call_price_panel,
                "put": put_price_panel,
                "long_strike_multiplier": long_strike_multiplier,
                "short_strike_multiplier": short_strike_multiplier,
            }
            instrument_return_panels[option_name] = {
                "long": call_price_panel.pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0),
                "short": put_price_panel.pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0),
            }

variant_runs = {}
summary_rows = []
yearly_rows = []

strategy_specs = []
if "classifier_prob" in strategy_variants or "raw_pct6" in strategy_variants:
    strategy_specs.append(
        {
            "strategy": "classifier_prob_long_only",
            "kind": "ml_long_only",
            "score_col": score_col,
            "component_cols": component_cols,
            "component_threshold": float(APP_CFG["strategy"]["component_threshold"]),
        }
    )
    strategy_specs.append(
        {
            "strategy": "classifier_prob_long_short",
            "kind": "ml_long_short",
            "long_score_col": score_col,
            "short_score_col": short_score_col,
            "long_component_cols": component_cols,
            "short_component_cols": short_component_cols,
            "component_threshold": float(APP_CFG["strategy"]["component_threshold"]),
        }
    )
if "momentum_21d" in strategy_variants:
    strategy_specs.append(
        {
            "strategy": "momentum_21d",
            "kind": "momentum",
            "lookback_days": int(APP_CFG["strategy"].get("baseline_lookback_days", 21)),
        }
    )
for family_name in family_models.keys():
    strategy_specs.append(
        {
            "strategy": f"family::{family_name}",
            "kind": "family_model",
            "family_name": family_name,
        }
    )

with timed_phase("strategy and backtest loop"):
    for strategy_spec in strategy_specs:
        strategy_name = str(strategy_spec["strategy"])
        spec_top_k_values = list(top_k_values)
        for top_k in spec_top_k_values:
            if strategy_spec["kind"] == "ml_long_only":
                signal_run = run_top_k_long_only_score_rule(
                    panel=bt_panel_5y,
                    score_col=str(strategy_spec["score_col"]),
                    component_cols=list(strategy_spec.get("component_cols", [])),
                    component_threshold=float(strategy_spec.get("component_threshold", 0.0)),
                    price_col="close",
                    top_k=int(top_k),
                    rebalance_freq=None,
                )
            elif strategy_spec["kind"] == "ml_long_short":
                signal_run = run_top_k_long_short_score_rule(
                    panel=bt_panel_5y,
                    long_score_col=str(strategy_spec["long_score_col"]),
                    short_score_col=str(strategy_spec["short_score_col"]),
                    long_component_cols=list(strategy_spec.get("long_component_cols", [])),
                    short_component_cols=list(strategy_spec.get("short_component_cols", [])),
                    component_threshold=float(strategy_spec.get("component_threshold", 0.0)),
                    price_col="close",
                    top_k=int(top_k),
                    rebalance_freq=None,
                )
            elif strategy_spec["kind"] == "family_model":
                family_name = str(strategy_spec["family_name"])
                family_scored = _build_single_family_prediction_panel(
                    family_name=family_name,
                    start_ts=BT_START_TS,
                    end_ts=BT_END_TS,
                    progress_logger=log_refresh,
                )
                if family_scored.empty:
                    continue

                family_panel = make_backtest_panel(
                    scored_panel=family_scored,
                    technical_df=bt_panel_5y,
                    start=BT_START_TS,
                    end=BT_END_TS,
                )
                family_panel = enrich_scored_panel(
                    family_panel,
                    prob_config=ProbabilityColumnConfig(
                        buy_col="clf__prob_1",
                        short_col=None,
                        infer_short_from_buy=True,
                    ),
                )
                family_panel = family_panel.replace([np.inf, -np.inf], np.nan)
                family_panel = family_panel.dropna(subset=["close", "prob_buy", "prob_short", "ranking", "ae_familiarity"])
                signal_run = run_top_k_long_short_score_rule(
                    panel=family_panel,
                    long_score_col="prob_buy",
                    short_score_col="prob_short",
                    long_component_cols=["prob_buy"],
                    short_component_cols=["prob_short"],
                    component_threshold=float(APP_CFG["strategy"]["component_threshold"]),
                    price_col="close",
                    top_k=int(top_k),
                    rebalance_freq=None,
                )
            else:
                signal_run = run_top_k_momentum_baseline(
                    panel=bt_panel_5y,
                    price_col="close",
                    top_k=int(top_k),
                    lookback_days=int(strategy_spec.get("lookback_days", 21)),
                    rebalance_freq=None,
                )

            positions = signal_run["positions"]
            position_state = shared_prepare_backtest_position_state(positions)
            prior_positions = positions.shift(1).fillna(0)
            buy_count = int(((positions == 1) & (prior_positions != 1)).sum().sum())
            sell_count = int(((prior_positions == 1) & (positions != 1)).sum().sum())
            short_count = int(((positions == -1) & (prior_positions != -1)).sum().sum())
            cover_count = int(((prior_positions == -1) & (positions != -1)).sum().sum())
            avg_active_names = float(positions.ne(0).sum(axis=1).mean()) if len(positions) else np.nan
            avg_long_names = float((positions == 1).sum(axis=1).mean()) if len(positions) else np.nan
            avg_short_names = float((positions == -1).sum(axis=1).mean()) if len(positions) else np.nan

            for instrument_name, asset_return_spec in instrument_return_panels.items():
                bt = shared_backtest_positions_with_directional_asset_returns(
                    positions,
                    asset_return_spec["long"],
                    short_asset_returns=asset_return_spec["short"],
                    initial_balance=100000.0,
                    fee_bps=float(APP_CFG["costs"]["fee_bps"]),
                    slippage_bps=float(APP_CFG["costs"]["slippage_bps"]),
                    position_state=position_state,
                )
                mode = f"{strategy_name}_{instrument_name}_top_{top_k}"
                curve_summary = summarize_curve(bt["returns"], years, mode=mode)
                variant_runs[(strategy_name, instrument_name, int(top_k))] = {
                    "positions": positions,
                    "backtest": bt,
                    "summary": curve_summary,
                }
                summary_rows.append(
                    {
                        "strategy": strategy_name,
                        "instrument": instrument_name,
                        "top_k": int(top_k),
                        "total_return_pct": curve_summary["total_return_pct"],
                        "sharpe": curve_summary["sharpe"],
                        "max_drawdown_pct": curve_summary["max_drawdown_pct"],
                        "buy_count": buy_count,
                        "sell_count": sell_count,
                        "short_count": short_count,
                        "cover_count": cover_count,
                        "avg_active_names": avg_active_names,
                        "avg_long_names": avg_long_names,
                        "avg_short_names": avg_short_names,
                        "avg_turnover": float(bt["turnover"].mean()) if len(bt["turnover"]) else np.nan,
                    }
                )
                yearly_df = curve_summary["yearly_df"].copy()
                yearly_df.insert(0, "strategy", strategy_name)
                yearly_df.insert(1, "instrument", instrument_name)
                yearly_df.insert(2, "top_k", int(top_k))
                yearly_rows.append(yearly_df)

summary_df = pd.DataFrame(summary_rows).sort_values(["strategy", "instrument", "top_k"]).reset_index(drop=True)
yearly_summary_df = pd.concat(yearly_rows, ignore_index=True)

print("Synthetic options comparison summary")
display(summary_df)

print("Yearly summary")
display(yearly_summary_df)


In [ ]:
summary_df.sort_values("sharpe", ascending=False)

In [ ]:
yearly_summary_df.groupby(["strategy", "instrument", "top_k"])[["total_return_pct", "max_drawdown_pct"]].describe()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

INSTRUMENT_NAMES = ["equity", "atm_option", "otm_option", "ditm_option"]
ML_STRATEGY_NAMES = ["classifier_prob_long_only", "classifier_prob_long_short"]


def get_growth_multiple(key):
    """
    Returns final growth multiple for a variant key:
    1.0 = flat
    2.0 = +100%
    0.5 = -50%
    """
    if key not in variant_runs:
        return float("-inf")

    eq = variant_runs[key]["summary"]["equity_curve"]

    if eq is None or len(eq) == 0:
        return float("-inf")

    base = max(float(eq.iloc[0]), 1e-12)
    final = float(eq.iloc[-1])

    return final / base


def get_total_return_pct(key):
    growth = get_growth_multiple(key)

    if not np.isfinite(growth):
        return float("-inf")

    return (growth - 1.0) * 100.0


def sort_keys_by_total_return(keys, reverse=True):
    return sorted(keys, key=get_total_return_pct, reverse=reverse)


def plot_variant_curves(
    variant_keys,
    *,
    title,
    label_fn=None,
    figsize=(12, 5),
    top_n=None,
):
    sorted_keys = sort_keys_by_total_return(variant_keys)

    if top_n is not None:
        sorted_keys = sorted_keys[:top_n]

    fig, ax = plt.subplots(figsize=figsize)
    plotted = 0

    for rank, key in enumerate(sorted_keys, start=1):
        if key not in variant_runs:
            continue

        eq = variant_runs[key]["summary"]["equity_curve"]

        if eq is None or len(eq) == 0:
            continue

        base = max(float(eq.iloc[0]), 1e-12)
        growth_curve = eq / base

        total_return_pct = get_total_return_pct(key)
        growth_multiple = get_growth_multiple(key)

        if label_fn is None:
            label = (
                f"#{rank} {key[0]} {key[1]} top_{key[2]} "
                f"| return={total_return_pct:,.1f}% "
                f"| {growth_multiple:,.2f}x"
            )
        else:
            label = label_fn(key, rank, total_return_pct, growth_multiple)

        growth_curve.plot(ax=ax, lw=2, label=label)
        plotted += 1

    ax.set_title(title)
    ax.set_xlabel("Date")
    ax.set_ylabel("Growth (start=1.0)")
    ax.grid(True, alpha=0.3)

    if plotted:
        ax.legend(title="Sorted by total return", fontsize=8)

    plt.show()


strategy_names = sorted(summary_df["strategy"].dropna().unique())
top_k_values = sorted(int(x) for x in summary_df["top_k"].dropna().unique())


# =========================
# Equity variants
# =========================

equity_keys = [
    (strategy_name, "equity", top_k)
    for strategy_name in strategy_names
    for top_k in top_k_values
]

plot_variant_curves(
    equity_keys,
    title="Equity Strategy Variants Sorted by Total Return",
    label_fn=lambda key, rank, ret, growth: (
        f"#{rank} {key[0]} equity top_{key[2]} "
        f"| return={ret:,.1f}% | {growth:,.2f}x"
    ),
)


# =========================
# OTM option variants
# =========================

otm_keys = [
    (strategy_name, "otm_option", top_k)
    for strategy_name in strategy_names
    for top_k in top_k_values
]

plot_variant_curves(
    otm_keys,
    title="OTM Option Strategy Variants Sorted by Total Return",
    label_fn=lambda key, rank, ret, growth: (
        f"#{rank} {key[0]} otm top_{key[2]} "
        f"| return={ret:,.1f}% | {growth:,.2f}x"
    ),
)


# =========================
# Optional: plot only top 5
# =========================

plot_variant_curves(
    otm_keys,
    title="Top 5 OTM Option Strategy Variants by Total Return",
    top_n=5,
)


In [ ]:
pd.DataFrame(yearly_summary_df.groupby(["strategy"])[["total_return_pct", "max_drawdown_pct"]].describe())